
# Random Forest Regressor Pipeline (Notebook Version)

This notebook replicates the core steps from `torch_prep_kfold.py` **without** K-folds, focused on a clear, linear pipeline:

1. Load two CSVs and merge on a key.
2. Train/test split **by sequence** (regression).
3. Keep **last 90%** of rows per sequence (based on temporal order: sequence → run → frame/time if available).
4. Average numeric features in groups of `navg=280` **per sequence**.
5. Standardize features using **train mean/std** (label unaffected); apply the same transform to test.
6. Train a **RandomForestRegressor** and evaluate (MSE, MAE, R²).
7. Save the fitted model & scaler.

> ⚙️ **You must configure the paths and column names in the first cell below.**


In [25]:
import os, math, json, random
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit, KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, matthews_corrcoef, accuracy_score
from scipy.stats import pearsonr


In [26]:

# --- CONFIG ---
DATA_FEATURES_CSV = "rawdat.csv"          # computed/processed features
DATA_LABELS_CSV   = "exp_data_all.csv"    # experimental labels
OUTPUT_DIR        = "outputs"             # where to save artifacts
# OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Column names (edit to match your files)
ID_COL      = "sequence"       # key column present in BOTH CSVs to merge on
LABEL_COL   = "bind_avg"     # the regression target
RUN_COL     = "run"          # optional temporal column 1 (e.g., run index)
FRAME_COLS  = []  # optional temporal column 2 (first found is used)

# Split / filtering / aggregation
TEST_SIZE        = 0.2       # fraction of sequences for test split
RANDOM_STATE     = 42
KEEP_LAST_PCT    = 90.0      # keep last X% per sequence
NAVG             = 280       # average numeric features in chunks of N rows per sequence

# Model
N_ESTIMATORS     = 300
MAX_DEPTH        = 10
N_JOBS           = -1

MAX_FEATURES     = "sqrt"
MIN_SAMPLES_SPLIT = 50        
MIN_SAMPLES_LEAF  = 20 

# Metrics to show
from pprint import pprint
print("Config loaded:")
pprint({
    "features_csv": DATA_FEATURES_CSV,
    "labels_csv": DATA_LABELS_CSV,
    "id_col": ID_COL,
    "label_col": LABEL_COL,
    "run_col": RUN_COL,
    "frame_cols": FRAME_COLS,
    "test_size": TEST_SIZE,
    "random_state": RANDOM_STATE,
    "keep_last_pct": KEEP_LAST_PCT,
    "navg": NAVG,
    "rf_params": {"n_estimators": N_ESTIMATORS, "max_depth": MAX_DEPTH, "n_jobs": N_JOBS},
})


Config loaded:
{'features_csv': 'rawdat.csv',
 'frame_cols': [],
 'id_col': 'sequence',
 'keep_last_pct': 90.0,
 'label_col': 'bind_avg',
 'labels_csv': 'exp_data_all.csv',
 'navg': 280,
 'random_state': 42,
 'rf_params': {'max_depth': 10, 'n_estimators': 300, 'n_jobs': -1},
 'run_col': 'run',
 'test_size': 0.2}


In [27]:
def load_features(path, id_col=ID_COL):
    df = pd.read_csv(path)
    if id_col not in df.columns:
        raise ValueError(f"{path} must contain '{id_col}'")
    return df

def load_reference(path, id_col=ID_COL, label_col=LABEL_COL):
    df = pd.read_csv(path)
    req = {id_col, label_col}
    if not req.issubset(df.columns):
        raise ValueError(f"{path} must contain {req}")
    return df[[id_col, label_col]]

feat_df = load_features(DATA_FEATURES_CSV, ID_COL)
ref_df  = load_reference(DATA_LABELS_CSV, ID_COL, LABEL_COL)

# Merge on sequence
merged = feat_df.merge(ref_df, on=ID_COL, how="inner").sample(frac=1.0, random_state=RANDOM_STATE).reset_index(drop=True)
print(merged.shape, merged.columns.tolist()[:12])


(68040, 11) ['sequence', 'run', 'VDWAALS', 'EEL', 'EGB', 'ESURF', 'HB Energy', 'Hydrophobic Energy', 'Pi-Pi Energy', 'Delta_Entropy', 'bind_avg']


In [28]:
# unique sequences → grouped split (no sequence leakage)
seq_ids = merged[ID_COL].values
gss = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(merged, groups=seq_ids))
df_train_raw = merged.iloc[train_idx].copy()
df_test_raw  = merged.iloc[test_idx].copy()

# optional: drop a "run" column if you have it
for df in (df_train_raw, df_test_raw):
    if "run" in df.columns:
        df.drop(columns=["run"], inplace=True, errors="ignore")

df_train_raw.shape, df_test_raw.shape


((53460, 10), (14580, 10))

In [29]:
def keep_last_n_percent(df, seq_col=ID_COL, keep_percent=KEEP_LAST_PCT):
    if keep_percent <= 0 or keep_percent >= 100: 
        return df.copy()
    # If you have a time/run column, sort by it; else stable order per sequence
    sort_cols = [seq_col] + (["run"] if "run" in df.columns else [])
    df_sorted = df.sort_values(sort_cols, kind="mergesort")
    # Compute per-sequence tail mask
    sizes = df_sorted.groupby(seq_col)[seq_col].transform("size")
    cum   = df_sorted.groupby(seq_col).cumcount()
    n_keep = (sizes * (keep_percent/100.0)).astype(int).clip(lower=1)
    mask = cum >= (sizes - n_keep)
    return df_sorted[mask].reset_index(drop=True)

df_train_filt = keep_last_n_percent(df_train_raw, ID_COL, KEEP_LAST_PCT)
df_test_filt  = keep_last_n_percent(df_test_raw,  ID_COL, KEEP_LAST_PCT)
df_train_filt.shape, df_test_filt.shape


((48114, 10), (13122, 10))

In [30]:
def average_sequence_chunks(df, id_col=ID_COL, label_col=LABEL_COL, navg=NAVG, seed=RANDOM_STATE):
    out = []
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    feat_cols = [c for c in num_cols if c not in [label_col]]
    # Note: id is not numeric; we add it back manually per chunk
    for seq, g in df.groupby(id_col):
        g = g.sample(frac=1.0, random_state=seed).reset_index(drop=True)
        n_chunks = len(g) // navg
        for i in range(n_chunks):
            ch = g.iloc[i*navg:(i+1)*navg]
            row = {c: ch[c].mean() for c in feat_cols}
            row[id_col] = seq
            row[label_col] = float(ch[label_col].iloc[0])
            out.append(row)
    if not out:
        # if navg is too large, you may end with no rows; consider lowering NAVG
        return pd.DataFrame(columns=[id_col, label_col] + feat_cols)
    cols = [id_col, label_col] + [c for c in feat_cols if c not in [id_col, label_col]]
    return pd.DataFrame(out)[cols]

df_train_avg = average_sequence_chunks(df_train_filt)
df_test_avg  = average_sequence_chunks(df_test_filt)
df_train_avg.shape, df_test_avg.shape


((165, 10), (45, 10))

In [31]:
def compute_stats(df, id_col=ID_COL, label_col=LABEL_COL):
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    for c in (label_col,):
        if c in num_cols: num_cols.remove(c)
    means = df[num_cols].mean()
    stds  = df[num_cols].std()
    return means, stds, num_cols

def apply_standardize(df, means, stds, cols):
    out = df.copy()
    for c in cols:
        mu, sd = means[c], stds[c]
        out[c] = (out[c] - mu) if (sd==0 or np.isnan(sd)) else (out[c] - mu)/sd
    return out

means, stds, feat_cols = compute_stats(df_train_avg, ID_COL, LABEL_COL)
df_train_std = apply_standardize(df_train_avg, means, stds, feat_cols)
df_test_std  = apply_standardize(df_test_avg,  means, stds, feat_cols)
df_train_std = df_train_std.sample(frac=1.0, random_state=RANDOM_STATE).reset_index(drop=True)
df_test_std  = df_test_std.sample(frac=1.0,  random_state=RANDOM_STATE).reset_index(drop=True)

df_train_std.shape, df_test_std.shape, feat_cols[:5]


((165, 10), (45, 10), ['VDWAALS', 'EEL', 'EGB', 'ESURF', 'HB Energy'])

In [32]:
def repeated_group_kfold_indices(groups, n_splits=5, n_repeats=5, base_seed=RANDOM_STATE):
    """
    Yield (repeat_idx, fold_idx, train_index_mask, val_index_mask) for each split.
    Splits by unique groups (sequence), shuffling group order per repeat.
    """
    groups = np.asarray(groups)
    N = len(groups)
    uniq = np.array(pd.unique(groups))
    for r in range(n_repeats):
        rng = np.random.default_rng(base_seed + 100*r)
        perm_groups = rng.permutation(uniq)
        # slice permuted groups into folds
        group_folds = np.array_split(perm_groups, n_splits)
        for f in range(n_splits):
            val_g = group_folds[f]
            val_mask = np.isin(groups, val_g)
            trn_mask = ~val_mask
            yield r, f, trn_mask, val_mask


In [33]:
# ----- CV plan -----
K_FOLDS     = 5
N_REPEATS   = 1

In [34]:
def rf_model():
    return RandomForestRegressor(
        n_estimators=N_ESTIMATORS,
        max_depth=MAX_DEPTH,
        max_features=MAX_FEATURES,
        min_samples_split=MIN_SAMPLES_SPLIT,
        min_samples_leaf=MIN_SAMPLES_LEAF,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

def eval_reg_metrics(y_true, y_pred, threshold=0.0):
    mse  = mean_squared_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred) if len(y_true) > 1 else np.nan
    pear = pearsonr(y_true, y_pred)[0] if len(y_true) > 1 else np.nan
    # optional: classification-style metrics by thresholding
    y_bin = (np.array(y_true) > threshold).astype(int)
    p_bin = (np.array(y_pred) > threshold).astype(int)
    if len(np.unique(y_bin)) > 1:
        mcc = matthews_corrcoef(y_bin, p_bin)
        acc = accuracy_score(y_bin, p_bin)
    else:
        mcc = acc = np.nan
    return dict(MSE=mse, R2=r2, Pear=pear, MCC=mcc, Accuracy=acc)

# Prepare arrays
feat_cols = [c for c in df_train_std.columns if c not in [ID_COL, LABEL_COL]]
X = df_train_std[feat_cols].values
y = df_train_std[LABEL_COL].values
g = df_train_std[ID_COL].values

cv_metrics = []
for rep, fold, trn_mask, val_mask in repeated_group_kfold_indices(g, n_splits=K_FOLDS, n_repeats=N_REPEATS):
    Xtr, ytr = X[trn_mask], y[trn_mask]
    Xva, yva = X[val_mask], y[val_mask]

    model = rf_model().fit(Xtr, ytr)
    yhat  = model.predict(Xva)

    # aggregate by sequence (mean) before scoring, like your scripts
    ids_va = g[val_mask]
    df_va  = pd.DataFrame({"id": ids_va, "y": yva, "p": yhat})
    agg    = df_va.groupby("id").agg(y=("y","mean"), p=("p","mean")).reset_index()
    m = eval_reg_metrics(agg["y"].values, agg["p"].values, threshold=0.0)
    m.update({"rep": rep, "fold": fold})
    cv_metrics.append(m)

# pd.DataFrame(cv_metrics).groupby([]).mean(numeric_only=True)
pd.DataFrame(cv_metrics)


,MSE,R2,Pear,MCC,Accuracy,rep,fold
0,0.928712,0.009529,0.211245,-0.258199,0.571429,0,0
1,0.382707,0.218576,0.522319,0.166667,0.571429,0,1
2,0.333781,0.501643,0.837423,0.730297,0.857143,0,2
3,0.437419,0.352517,0.904624,0.500000,0.666667,0,3
4,0.617021,-0.313179,0.100010,0.500000,0.666667,0,4


In [35]:
# cv_metrics is a list of dicts like {"MSE": ..., "R2": ..., "Pear": ..., "MCC": ..., "Accuracy": ..., "rep": ..., "fold": ...}
cv_df = pd.DataFrame(cv_metrics)

# 1) Overall averages (no grouping)
overall_mean = cv_df.mean(numeric_only=True)
overall_std  = cv_df.std(numeric_only=True)
print("Overall mean:\n", overall_mean[['MSE','R2','Pear','MCC','Accuracy']])
print("Overall std:\n",  overall_std[['MSE','R2','Pear','MCC','Accuracy']])

Overall mean:
 MSE         0.539928
R2          0.153817
Pear        0.515124
MCC         0.327753
Accuracy    0.666667
dtype: float64
Overall std:
 MSE         0.242291
R2          0.317633
Pear        0.360660
MCC         0.384275
Accuracy    0.116642
dtype: float64


In [36]:
# 8)

# Train on full standardized & averaged TRAIN set
final_model = rf_model().fit(X, y)

# Prepare standardized & averaged TEST set arrays
X_test = df_test_std[feat_cols].values
y_test = df_test_std[LABEL_COL].values
ids_t  = df_test_std[ID_COL].values

yhat_t = final_model.predict(X_test)
df_t   = pd.DataFrame({"id": ids_t, "y": y_test, "p": yhat_t})
agg_t  = df_t.groupby("id").agg(y=("y","mean"), p=("p","mean")).reset_index()
test_metrics = eval_reg_metrics(agg_t["y"].values, agg_t["p"].values, threshold=0.0)
test_metrics


{'MSE': 0.3260852509391501,
 'R2': 0.366677314906111,
 'Pear': 0.7044633643394319,
 'MCC': 0.55,
 'Accuracy': 0.7777777777777778}

In [38]:

# 9) (Optional) Save artifacts like the CLI does

from joblib import dump

os.makedirs("Model", exist_ok=True)
dump(final_model, "Model/rf_final_reg.joblib")

pd.DataFrame(cv_metrics).to_csv("Random_Forest_metrics_cv.csv", index=False)
pd.DataFrame({"Label": agg_t["id"], "AvgTrue": agg_t["y"], "AvgPredicted": agg_t["p"]}).to_csv(
    "predictions_reg_test_final.csv", index=False
)

with open("best_hyperparams_reg.json", "w") as f:
    json.dump({
        "keep_last_percent": KEEP_LAST_PCT,
        "navg": NAVG,
        "n_estimators": N_ESTIMATORS,
        "max_depth": MAX_DEPTH,
        "max_features": MAX_FEATURES,
        "min_samples_split": MIN_SAMPLES_SPLIT,
        "min_samples_leaf": MIN_SAMPLES_LEAF
    }, f, indent=2)
    


In [39]:
import matplotlib.pyplot as plt

# assume you stored all validation predictions like this:
# df_va  = pd.DataFrame({"id": ids_va, "y": yva, "p": yhat})
# agg    = df_va.groupby("id").agg(y=("y","mean"), p=("p","mean")).reset_index()
# and you appended agg.assign(rep=rep, fold=fold) to a list called all_val_preds

all_val_preds = pd.concat(all_val_preds, ignore_index=True)

plt.figure(figsize=(6,6))
plt.scatter(all_val_preds["y"], all_val_preds["p"], s=25, alpha=0.6)
plt.xlabel("True ΔΔG (kcal/mol)")
plt.ylabel("Predicted ΔΔG (kcal/mol)")
plt.title("Random Forest Regression (5×5 CV)")
# y=x line
lims = [min(all_val_preds[["y","p"]].min()) , max(all_val_preds[["y","p"]].max())]
plt.plot(lims, lims, 'r--', lw=1)
plt.xlim(lims); plt.ylim(lims)
plt.grid(True, linestyle='--', alpha=0.3)
plt.show()


NameError: name 'all_val_preds' is not defined

In [ ]:
plt.figure(figsize=(6,6))
plt.scatter(agg_t["y"], agg_t["p"], s=25, alpha=0.6, color='darkblue')
plt.plot(lims, lims, 'r--')
plt.xlabel("True ΔΔG (test)")
plt.ylabel("Predicted ΔΔG (test)")
plt.title("Random Forest – Test Set")
plt.grid(True, linestyle='--', alpha=0.3)
plt.show()
